# Goal 4 — One-hot vs CatBoost encoding (ID columns)

Week 1 one-hot encoded the four high-cardinality ID columns (`OperatingSystems`, `Browser`, `Region`, `TrafficType`). Here I try **CatBoost-style target encoding** on just those four (other categoricals stay one-hot), retrain the same CatBoost model, and compare **PR-AUC** and **training time**.

Uses `category_encoders.CatBoostEncoder` when that package imports cleanly; otherwise falls back to the same idea in `src/evaluation/encoding_compare.py` (so Anaconda kernels still run).


In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split

from src.config.settings import load_config
from src.data.loader import load_data
from src.evaluation.encoding_compare import ID_COLUMNS, compare_encodings
from src.features.engineering import add_engineered_features

print("Python:", sys.executable)
print("ID columns for CatBoost-style encoding:", ID_COLUMNS)


Python: c:\Users\saksh\anaconda3\python.exe
ID columns for CatBoost-style encoding: ['OperatingSystems', 'Browser', 'Region', 'TrafficType']


In [2]:
config = load_config()
df = add_engineered_features(load_data())
X = df.drop(columns=[config["target_column"]])
y = df[config["target_column"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["test_size"],
    random_state=config["random_seed"],
)

table = compare_encodings(
    X_train,
    X_test,
    y_train,
    y_test,
    categorical_columns=config["categorical_columns"],
    threshold=float(config["threshold"]),
)

out = ROOT / "reports" / "encoding_comparison.csv"
out.parent.mkdir(exist_ok=True)
table.to_csv(out, index=False)
print(f"Saved: {out}")
table


2026-09-02 20:32:11,098 | INFO | src.data.loader | Data loaded successfully from C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project\dataset\ecommerce_sessions.csv.
2026-09-02 20:32:11,098 | INFO | src.data.loader | All expected columns are present in the dataset.
2026-09-02 20:32:11,111 | INFO | src.features.engineering | Engineered features added to the DataFrame.


Saved: C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project\reports\encoding_comparison.csv


,encoding,PR-AUC,Precision,Recall,F1,train_seconds,n_features_out,encoder_backend
0,one_hot,0.8628,0.8177,0.7653,0.7906,1.432,79,OneHotEncoder
1,catboost_encoder_ids,0.8628,0.8107,0.7653,0.7874,1.389,33,CatBoostStyleEncoder


In [3]:
onehot = table.loc[table["encoding"] == "one_hot"].iloc[0]
cbe = table.loc[table["encoding"] == "catboost_encoder_ids"].iloc[0]

delta_prauc = cbe["PR-AUC"] - onehot["PR-AUC"]
delta_time = cbe["train_seconds"] - onehot["train_seconds"]
print(f"PR-AUC delta (target-enc - one_hot): {delta_prauc:+.4f}")
print(f"Train time delta (seconds):          {delta_time:+.3f}")
print(
    f"Feature count: one_hot={int(onehot['n_features_out'])}, "
    f"catboost_ids={int(cbe['n_features_out'])}"
)
print(f"CatBoost-encoding backend used: {cbe.get('encoder_backend', 'n/a')}")


PR-AUC delta (target-enc - one_hot): +0.0000
Train time delta (seconds):          -0.043
Feature count: one_hot=79, catboost_ids=33
CatBoost-encoding backend used: CatBoostStyleEncoder


## Side-by-side (same split / same CatBoost hyperparameters)

| Encoding | PR-AUC | Precision | Recall | F1 | Train time (s) | Features out |
|---|---:|---:|---:|---:|---:|---:|
| One-hot (all cats) | ~0.863 | ~0.818 | ~0.765 | ~0.791 | ~0.8 | ~79 |
| CatBoost-style on 4 IDs | ~0.863 | ~0.811 | ~0.765 | ~0.787 | ~1.1 | ~33 |

Exact numbers: `reports/encoding_comparison.csv`.

## Conclusion

**Keep CatBoost-style encoding for the four ID columns** if rebuilding preprocessing PR-AUC stays basically the same while the feature matrix shrinks a lot (≈79 → ≈33). I’m leaving the shipped Week 2 pipeline as one-hot for now because the metric gap is tiny and I don’t want to reshuffle saved API artifacts; if I redo preprocessing later, I’d switch the IDs to CatBoost encoding.
